# Team 23 – Decode the Bioreactor (v34)

This notebook predicts **VCD, TCD, substrate and byproduct** for days 1-13 of the 12 test batches.
For the test batches we only know the day-0 measurements, the full feed plan and the volume.

What is new in v34 compared with v31:
1. **Substrate:** look-ahead features. The model sees the time and size of the *next* substrate feed. These are known in advance from the feed plan, so they are allowed inputs.
2. **Byproduct:** the gradient boosting model is trained with `sample_weight = 1/y`. MAPE punishes errors on small values the most, so the weighting makes the model focus on them.

Evaluation metric: MAPE per target and the mean of the four.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import argparse
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.covariance import LedoitWolf
from sklearn.linear_model import Ridge

## 2. Paths

Put `bioreaction_remaining.csv` and `bioreaction_template.csv` in the same folder as this notebook, or change the paths below.

In [ ]:
RENKU = Path('/home/renku/work/team-workspace')
BASE_DIR = RENKU if (RENKU / 'bioreaction_remaining.csv').exists() else Path.cwd()
DATA = str(BASE_DIR / 'bioreaction_remaining.csv')
TEMPLATE = str(BASE_DIR / 'bioreaction_template.csv')
OUTPUT = str(BASE_DIR / 'TEAM23_submission_v34.csv')

## 3. Configuration

- `C_FEED`: substrate content of each feed (feed_substrate, feed_b, feed_c, feed_d). It is used in the mass balance feature `snc`.
- `S_TRIG`, `S_LOW`: substrate thresholds of the plant controller. The size of a substrate shot tells us if the substrate was above or below the threshold.
- `P0_KAPPA`: shrinks the day-0 byproduct of test batches toward the training mean.
- `CFG`: features, ridge penalty, fused penalty across days, boosting share and Huber delta for each target (order: VCD, TCD, substrate, byproduct).

**Bug fix:** in the original code `gfeats[2]` was `None` while `gmix[2] = 0.20`. This crashes with `TypeError: 'NoneType' object is not iterable`. Here the substrate feature list is used for the substrate boosting model.

In [ ]:
P0_KAPPA = 0.40

TARGETS = ["vcd", "tcd", "substrate", "byproduct"]
FEEDS = ["feed_substrate", "feed_b", "feed_c", "feed_d"]
C_FEED = np.array([197.0, 70.4, 0.0, 38.2])
S_TRIG, S_LOW = 1.95, 0.965

BASE = ["y0", "v0", "dvol", "t"]
CLIP = [False, False, True, True]
SHRINK_TAU = 0.3

CFG = dict(
    feats=[BASE + ["cum_bd", "snc_tile", "ly0"],
           BASE + ["cum_bd", "snc_tile", "ly0"],
           # Substrate: Look-Ahead özellikleri eklendi
           ["y0", "dvol", "t", "since", "lsnc", "snc", "snc_tile", "un_cum", "low", "fb_now", "un_sum", 
            "last_feed_amount", "since_feed_amount", "next_feed_amt", "next_feed_time"],
           BASE + ["cum", "since", "snc_tile", "fb_now", "un_early", "nsmall", "un_next"]],
    alpha=[3, 3, 10, 3],
    fuse=[(10000, False), (10000, False), None, (30, True)],
    rw=[1.0, 1.0, 1.0, 1.0],
    gfeats=[BASE, BASE, ["y0", "dvol", "t", "since", "lsnc", "snc", "snc_tile", "un_cum", "low", "fb_now", "un_sum", "last_feed_amount", "since_feed_amount", "next_feed_amt", "next_feed_time"], BASE + ["un_sum", "fbf", "vcd_pred", "vcd_growth", "tcd_pred", "sub_pred", "cum_vcd", "vcd_lag", "sub_lag", "time_since_sub_low", "vcd_decay"]],
    gmix=[0.15, 0.15, 0.20, 0.40], # Substrate GBR payı 0.20'ye çıkarıldı (Look-ahead ile mükemmel çalışacak)
    drop=[True, True, False, True],
    robust_delta=[0.04, 0.07, None, 0.10],
)

BYPRODUCT_FEATS = [nm for nm in CFG["feats"][3] if nm not in ("cum", "snc_tile", "un_next")] + [
    "sub_ctrl", "fb_signal", "u_fut3", "vcd_pred", "vcd_growth", "tcd_pred", "sub_pred", 
    "cum_vcd", "vcd_lag", "sub_lag", "time_since_sub_low", "vcd_decay"
]
BYPRODUCT_BOOST_SHARE = 0.45

## 4. Data preparation

`prepare` builds one dictionary per batch. Main quantities:
- `cum`: cumulative amount of each feed before each sample time.
- `since`, `last_feed_amount`: time since the last substrate feed and its size.
- `next_feed_amt`, `next_feed_time`: **look-ahead** features from the feed plan.
- `snc`: mass-balance substrate concentration, `(S0·V0 + Σ feed × content) / V`.
- `trig_day`, `low11`, `low12`: controller flags read from the substrate shot sizes.
- `fb_dev`: log deviation of the feed B rate from the median schedule.
- `un`: unexplained volume change per day (base addition for pH control). It tracks byproduct production.

In [ ]:
def prepare(path):
    d = pd.read_csv(path)
    test_ids = set(d.loc[d.to_predict, "batch_id"])
    batches = []
    for bid, g in d.groupby("batch_id", sort=False):
        g = g.sort_values("elapsed_time")
        m = g.dropna(subset=TARGETS)
        q = g[g.to_predict] if bid in test_ids else m.iloc[1:]
        t = q.elapsed_time.to_numpy()
        init = m.iloc[0]
        cum = np.array([[g.loc[g[f].notna() & (g.elapsed_time < x), f].sum() for f in FEEDS] for x in t])
        fs = g[g.feed_substrate.notna()]
        sub_times = fs.elapsed_time.to_numpy()
        
        last_feed_amount = np.array([fs.loc[fs.elapsed_time < x, 'feed_substrate'].iloc[-1] if (fs.elapsed_time < x).any() else 0.0 for x in t])
        since = np.array([x - sub_times[sub_times < x].max() if (sub_times < x).any() else 15.0 for x in t])
        
        # YENİ: GELECEK BESLEME (LOOK-AHEAD)
        next_feed_amt = np.zeros_like(t)
        next_feed_time = np.full_like(t, 15.0)
        for i, x in enumerate(t):
            future_feeds = fs[fs.elapsed_time >= x]
            if len(future_feeds) > 0:
                next_feed_amt[i] = future_feeds.feed_substrate.iloc[0]
                next_feed_time[i] = future_feeds.elapsed_time.iloc[0] - x
        
        y0 = init[TARGETS].to_numpy(float)
        v0 = float(init.volume)

        vt = g.dropna(subset=["volume"])
        vgrid, vvals = vt.elapsed_time.to_numpy(), vt.volume.to_numpy()
        vol_at = lambda x: np.interp(x, vgrid, vvals)

        early = fs[fs.elapsed_time < 10.5]
        trig_day = int(np.floor(early.elapsed_time.iloc[0])) if len(early) else 0

        def low_flag(lo, hi):
            e = fs[(fs.elapsed_time > lo) & (fs.elapsed_time < hi)]
            if len(e) == 0: return np.nan
            return float(e.feed_substrate.iloc[0] / vol_at(e.elapsed_time.iloc[0]) > 0.005)

        fb = g[g.feed_b.notna()]
        fb_day = np.searchsorted(vgrid, fb.elapsed_time.to_numpy(), "right") - 1
        fb_rate = fb.feed_b.to_numpy() / vol_at(fb.elapsed_time.to_numpy())

        fc = g[g.feed_c.notna()]
        small_t = fc.loc[fc.feed_c < 0.0013, "elapsed_time"].to_numpy()

        allfeed = g[FEEDS].fillna(0).sum(1).to_numpy()
        et = g.elapsed_time.to_numpy()
        un = np.array([vvals[i + 1] - vvals[i] - allfeed[(et >= vgrid[i]) & (et < vgrid[i + 1])].sum()
                       for i in range(len(vgrid) - 1)])

        batches.append(dict(
            id=bid, test=bid in test_ids, t=t, y0=y0, v0=v0, vol=q.volume.to_numpy(), cum=cum, since=since,
            last_feed_amount=last_feed_amount, next_feed_amt=next_feed_amt, next_feed_time=next_feed_time,
            snc=(y0[2] * v0 + (cum * C_FEED).sum(1)) / q.volume.to_numpy(),
            trig_day=trig_day, low11=low_flag(11, 12), low12=low_flag(12, 13),
            fb_day=fb_day, fb_rate=fb_rate, small_t=small_t, un=un, vgrid=vgrid,
            y=None if bid in test_ids else q[TARGETS].to_numpy(float),
            ref_t=m.elapsed_time.to_numpy(), ref_y=m[TARGETS].to_numpy(float)))

    med = {k: np.median([b["fb_rate"][b["fb_day"] == k][0] for b in batches if (b["fb_day"] == k).any()])
           for k in range(1, 13)}
    for b in batches:
        dev = np.full(13, np.nan)
        for k in range(1, 13):
            r = b["fb_rate"][b["fb_day"] == k]
            if len(r): dev[k] = np.log(r[0] / med[k])
        b["fb_dev"] = dev
        b["blk"] = feature_blocks(b)
    return batches

## 5. Feature blocks

`feature_blocks` turns the batch dictionary into named feature matrices (one row per sample day). `X_of` stacks the selected blocks into a design matrix.

In [ ]:
def feature_blocks(b):
    n = len(b["t"])
    tile = lambda v: np.tile(np.atleast_1d(v), (n, 1))
    t = b["t"]
    k = np.round(t).astype(int)
    dev, un = b["fb_dev"], b["un"]
    l11 = 0.5 if np.isnan(b["low11"]) else b["low11"]
    l12 = 0.5 if np.isnan(b["low12"]) else b["low12"]
    
    sub_low_idx = np.where(b["snc"] < 1.5)[0]
    t_since_sub_low = np.zeros(n)
    if len(sub_low_idx) > 0:
        first_low = sub_low_idx[0]
        t_since_sub_low[first_low:] = t[first_low:] - t[first_low]

    return dict(
        y0=tile(b["y0"]), ly0=tile(np.log(b["y0"])), v0=tile(b["v0"]),
        dvol=(b["vol"] - b["v0"])[:, None], t=t[:, None],
        cum=b["cum"], cum_bd=b["cum"][:, [1, 3]], since=b["since"][:, None],
        last_feed_amount=b["last_feed_amount"][:, None],
        since_feed_amount=(b["since"] * b["last_feed_amount"])[:, None],
        next_feed_amt=b["next_feed_amt"][:, None],       # YENİ
        next_feed_time=b["next_feed_time"][:, None],     # YENİ
        snc=b["snc"][:, None], lsnc=np.log(b["snc"])[:, None], snc_tile=tile(b["snc"]),
        low=tile(np.array([l11, l12])),
        fbf=tile(np.array([np.nanmean(dev[1:6]), np.nanmean(dev[6:13]), np.nanmean(dev[1:13])])),
        fb_now=np.array([dev[kk] if 1 <= kk <= 12 else np.nanmean(dev[8:13]) for kk in k])[:, None],
        un_cum=np.interp(t, b["vgrid"], np.concatenate([[0], np.cumsum(un)]))[:, None],
        un_early=tile(un[1:5]),
        un_next=np.r_[un[1:], un[-1]][:, None],
        un_sum=tile(np.array([un[1:5].sum(), un[5:].sum()])),
        nsmall=np.column_stack([[np.sum(b["small_t"] < x) for x in t], np.full(n, len(b["small_t"]))]).astype(float),
        sub_ctrl=tile(np.array([b["trig_day"], l11, l12])),
        fb_signal=np.where(np.isfinite(dev[1:]), dev[1:], 0.0)[np.clip(k - 1, 0, 11), None],
        u_fut3=(un[np.clip(k, 0, 12)] + un[np.clip(k + 1, 0, 12)]
                + un[np.clip(k + 2, 0, 12)])[:, None] / 3,
        time_since_sub_low=t_since_sub_low[:, None]
    )


def X_of(b, names):
    return np.column_stack([b["blk"][nm] for nm in names])

## 6. Baselines and helpers

- `weighted_median`: weighted median of the training curves (weights `1/y`). This is the MAPE-optimal population curve.
- `jump_mask`: marks one-off late jumps (e.g. a sudden day-13 byproduct jump) so they do not distort training.

In [ ]:
def weighted_median(b, refs):
    prof = np.stack([np.column_stack([np.interp(b["t"], r["ref_t"], r["ref_y"][:, j]) for j in range(4)])
                     for r in refs])
    change = prof - np.stack([r["ref_y"][0] for r in refs])[:, None, :]
    w = 1 / np.maximum(np.abs(prof), 1e-8)
    order = np.argsort(change, 0)
    cw = np.take_along_axis(w, order, 0).cumsum(0)
    idx = (cw >= cw[-1] / 2).argmax(0)
    return b["y0"] + np.take_along_axis(np.take_along_axis(change, order, 0), idx[None], 0)[0]


def jump_mask(train, j):
    M = np.zeros((len(train), 13), bool)
    for i, b in enumerate(train):
        if j == 3 and b["y"][-1, 3] / b["y"][-2, 3] > 1.5:
            M[i, -1] = True
        if j in (0, 1):
            y = np.log(b["y"][:, j])
            if y[-1] - (2 * y[-2] - y[-3]) > 0.4:
                M[i, -1] = True
    return M

## 7. Regression models

All models work on **log targets**, so errors behave like relative errors.
- `ridge_per_step`: a separate ridge model for each day.
- `fused_ridge`: one ridge model per day, but neighbouring days are pushed to have similar coefficients (smooth over time).
- `boosting`: gradient boosting on all days pooled. With `use_weights=True` each sample gets weight `1/y` (new in v34).
- `robust_fused`: fused ridge with Huber-type reweighting (IRLS). Large residuals get less weight.

In [ ]:
def ridge_per_step(train, test, j, names, alpha, drop=None):
    X = np.stack([X_of(b, names) for b in train])
    Xt = np.stack([X_of(b, names) for b in test])
    Y = np.log(np.stack([b["y"][:, j] for b in train]))
    if CLIP[j]: Xt = np.clip(Xt, X.min(0), X.max(0))
    P = np.zeros((len(test), X.shape[1]))
    for s in range(X.shape[1]):
        ok = np.ones(len(train), bool) if drop is None else ~drop[:, s]
        c, sd = X[ok, s].mean(0), X[ok, s].std(0)
        sd[sd < 1e-8] = 1
        model = Ridge(alpha=alpha).fit((X[ok, s] - c) / sd, Y[ok, s])
        P[:, s] = np.exp(model.predict((Xt[:, s] - c) / sd))
    return P


def fused_ridge(train, test, j, names, alpha, lam, drop=None, day_scale=False):
    X = np.stack([X_of(b, names) for b in train])
    Xt = np.stack([X_of(b, names) for b in test])
    Y = np.log(np.stack([b["y"][:, j] for b in train]))
    if CLIP[j]: Xt = np.clip(Xt, X.min(0), X.max(0))
    n, S, p = X.shape
    W = np.ones((n, S)) if drop is None else (~drop).astype(float)
    if day_scale:
        mu = (X * W[..., None]).sum(0) / W.sum(0)[:, None]
        sd = np.sqrt(((X - mu) ** 2 * W[..., None]).sum(0) / W.sum(0)[:, None])
    else:
        mu = (X * W[..., None]).sum((0, 1)) / W.sum()
        sd = np.sqrt(((X - mu) ** 2 * W[..., None]).sum((0, 1)) / W.sum())
    sd = np.where(sd < 1e-8, 1, sd)
    Z, Zt = (X - mu) / sd, (Xt - mu) / sd
    A = np.zeros((S * p, S * p)); rhs = np.zeros(S * p)
    zbar = np.zeros((S, p)); ybar = np.zeros(S)
    for s in range(S):
        w = W[:, s]
        zbar[s] = (Z[:, s] * w[:, None]).sum(0) / w.sum(); ybar[s] = (Y[:, s] * w).sum() / w.sum()
        Zc = (Z[:, s] - zbar[s]) * np.sqrt(w)[:, None]; yc = (Y[:, s] - ybar[s]) * np.sqrt(w)
        blk = slice(s * p, (s + 1) * p)
        A[blk, blk] += Zc.T @ Zc + alpha * np.eye(p); rhs[blk] = Zc.T @ yc
    I = lam * np.eye(p)
    for s in range(S - 1):
        a, b_ = slice(s * p, (s + 1) * p), slice((s + 1) * p, (s + 2) * p)
        A[a, a] += I; A[b_, b_] += I; A[a, b_] -= I; A[b_, a] -= I
    B = np.linalg.solve(A, rhs).reshape(S, p)
    return np.exp(ybar[None] + np.einsum("nsp,sp->ns", Zt - zbar[None], B))


def boosting(train, test, j, names, drop=None, seeds=(0, 1, 2), use_weights=False):
    X = np.concatenate([X_of(b, names) for b in train])
    Xt = np.concatenate([X_of(b, names) for b in test])
    Y = np.log(np.concatenate([b["y"][:, j] for b in train]))
    ok = np.ones(len(Y), bool) if drop is None else ~drop.ravel()
    mu, sd = X.mean(0), X.std(0) + 1e-9
    
    # YENİ: MAPE Odaklı Ağırlıklandırma
    weights = None
    if use_weights:
        weights = 1.0 / np.exp(Y) # y değeri küçüldükçe ağırlık artar
        weights = weights / weights.mean()
    
    preds = []
    for seed in seeds:
        model = GradientBoostingRegressor(n_estimators=400, learning_rate=0.03, max_depth=3,
                                          subsample=0.8, random_state=seed)
        model.fit(((X - mu) / sd)[ok], Y[ok], sample_weight=weights[ok] if weights is not None else None)
        preds.append(model.predict((Xt - mu) / sd))
    return np.exp(np.mean(preds, 0)).reshape(len(test), -1)


def robust_fused(train, test, j, names, alpha, lam, delta, iterations=4, day_scale=False, clip_inputs=None):
    X=np.stack([X_of(b,names) for b in train]);Xt=np.stack([X_of(b,names) for b in test]);Y=np.log(np.stack([b['y'][:,j] for b in train]));n,S,k=X.shape
    if CLIP[j] and clip_inputs is not False: Xt=np.clip(Xt,X.min(0),X.max(0))
    M=jump_mask(train,j); W=(~M).astype(float)
    if day_scale:
        mu=(X*W[:,:,None]).sum(0)/W.sum(0)[:,None]
        sd=np.sqrt(((X-mu)**2*W[:,:,None]).sum(0)/W.sum(0)[:,None])
    else:
        mu=(X*W[:,:,None]).sum((0,1))/W.sum();sd=np.sqrt(((X-mu)**2*W[:,:,None]).sum((0,1))/W.sum())
    sd=np.where(sd<1e-8,1,sd);Z=(X-mu)/sd;Zt=(Xt-mu)/sd
    for it in range(iterations):
        A=np.zeros((S*k,S*k));rhs=np.zeros(S*k);ybar=np.zeros(S);zbar=np.zeros((S,k))
        for s in range(S):
            w=W[:,s]; zbar[s]=(Z[:,s]*w[:,None]).sum(0)/w.sum();ybar[s]=(Y[:,s]*w).sum()/w.sum()
            Zc=(Z[:,s]-zbar[s])*np.sqrt(w)[:,None];yc=(Y[:,s]-ybar[s])*np.sqrt(w)
            sl=slice(s*k,(s+1)*k);A[sl,sl]+=Zc.T@Zc+alpha*np.eye(k);rhs[sl]=Zc.T@yc
        I=lam*np.eye(k)
        for s in range(S-1):
            a=slice(s*k,(s+1)*k);b=slice((s+1)*k,(s+2)*k)
            A[a,a]+=I;A[b,b]+=I;A[a,b]-=I;A[b,a]-=I
        B=np.linalg.solve(A,rhs).reshape(S,k)
        if it<iterations-1:
            err=Y-ybar[None]-np.einsum('nsp,sp->ns',Z-zbar[None],B)
            ww=np.minimum(1.,delta/np.maximum(abs(err),1e-8)); W=(~M)*ww
    return np.exp(ybar[None]+np.einsum('nsp,sp->ns',Zt-zbar[None],B))

## 8. Ridge part per target

In [ ]:
def ridge_part(train, test, j, med):
    drop = jump_mask(train, j) if CFG["drop"][j] else None
    names = CFG["feats"][j] + (["nsmall"] if j in (0, 1) else [])
    if CFG["fuse"][j]:
        lam, day_scale = CFG["fuse"][j]
        if CFG["robust_delta"][j] is not None:
            r = robust_fused(train, test, j, names, CFG["alpha"][j], lam, CFG["robust_delta"][j], iterations=4, day_scale=day_scale)
        else: r = fused_ridge(train, test, j, names, CFG["alpha"][j], lam, drop, day_scale)
    else: r = ridge_per_step(train, test, j, names, CFG["alpha"][j], drop)
    if CFG["rw"][j] < 1: r = CFG["rw"][j] * r + (1 - CFG["rw"][j]) * med
    return np.maximum(r, 1e-6)

## 9. Substrate conditioning on controller information

The controller reveals if the substrate was above or below a threshold on some days. We use this as a constraint:
1. `substrate_cov` estimates the covariance of the log residuals across days (out-of-fold, Ledoit-Wolf).
2. `conditioned` samples residuals from a Gaussian, keeps only samples that satisfy the constraints and takes the `1/y`-weighted median.

In [ ]:
def constraints(b):
    out = []; td = b["trig_day"]
    for day in (7, 8, 9):
        if td == 0 or day < td: out.append((day - 1, S_TRIG, False))
        elif day == td: out.append((day - 1, S_TRIG, True))
    for day, flag in ((11, b["low11"]), (12, b["low12"])):
        if not np.isnan(flag): out.append((day - 1, S_LOW, bool(flag)))
    return out


def substrate_cov(train, folds=5):
    n = len(train); R = np.zeros((n, 13))
    for f in range(folds):
        vi = np.arange(f, n, folds); fi = np.setdiff1d(np.arange(n), vi)
        tr, va = [train[i] for i in fi], [train[i] for i in vi]
        med = np.stack([weighted_median(b, tr) for b in va])[..., 2]
        R[vi] = np.log(np.stack([b["y"][:, 2] for b in va]) / ridge_part(tr, va, 2, med))
    return LedoitWolf().fit(R).covariance_


def conditioned(p, cons, cov, rng, n=100000):
    L = np.linalg.cholesky(cov + 1e-9 * np.eye(len(p)))
    r = rng.standard_normal((n, len(p))) @ L.T; ok = np.ones(n, bool)
    for k, thr, below in cons:
        c = np.log(thr / p[k]); ok &= (r[:, k] < c) if below else (r[:, k] >= c)
    if ok.sum() < 200: ok[:] = True
    y = p[None] * np.exp(r[ok]); order = np.argsort(y, 0); ys = np.take_along_axis(y, order, 0)
    cw = (1 / ys).cumsum(0); idx = (cw >= cw[-1] / 2).argmax(0)
    return ys[idx, np.arange(len(p))]

## 10. Core prediction (v9 block)

For each target: ridge part + boosting share. Then substrate conditioning. VCD and TCD deviations from the population curve are softly limited with `tanh`, and TCD is forced to be at least VCD.

In [ ]:
def predict_v9(train, test):
    med = np.stack([weighted_median(b, train) for b in test]); P = np.zeros_like(med)
    for j in range(4):
        drop = jump_mask(train, j) if CFG["drop"][j] else None; r = ridge_part(train, test, j, med[..., j])
        if CFG["gmix"][j] > 0:
            r = (1 - CFG["gmix"][j]) * r + CFG["gmix"][j] * boosting(train, test, j, [name for name in CFG["gfeats"][j] if name not in {"vcd_pred", "vcd_growth", "tcd_pred", "sub_pred", "cum_vcd", "vcd_lag", "sub_lag", "time_since_sub_low", "vcd_decay"}], drop)
        P[..., j] = r
    
    cov = 1.5 * substrate_cov(train); rng = np.random.default_rng(123)
    for i, b in enumerate(test): P[i, :, 2] = conditioned(P[i, :, 2], constraints(b), cov, rng)
        
    for j in (0, 1):
        pop = np.exp(np.log(np.stack([b["y"][:, j] for b in train])).mean(0)); d = np.log(P[..., j] / pop)
        if SHRINK_TAU: d = SHRINK_TAU * np.tanh(d / SHRINK_TAU)
        P[..., j] = pop * np.exp(d)
    P[..., 1] = np.maximum(P[..., 1], P[..., 0])
    return P

## 11. Improved byproduct model

The byproduct model is stacked on the other targets:
- Extra features from VCD, TCD and substrate (growth, lags, cumulative VCD, decay after the peak, time since low substrate).
- The target is **byproduct per cell** (`byproduct / VCD`). The prediction is multiplied back by the predicted VCD.
- Final value: 55% robust fused ridge + 45% weighted boosting.

Note: during training the stacked features use the *true* VCD/TCD/substrate. At test time they use the *predicted* values.

In [ ]:
def improved_byproduct(train, test, vcd_train, tcd_train, sub_train, vcd_test, tcd_test, sub_test):
    def inject_stack_features(batch_list, vcd_arr, tcd_arr, sub_arr):
        out = []
        for i, b in enumerate(batch_list):
            c = b.copy()
            vcd = vcd_arr[i]; tcd = tcd_arr[i]; sub = sub_arr[i]
            
            vcd_growth = np.zeros_like(vcd); vcd_growth[1:] = np.diff(vcd)
            cum_vcd = np.cumsum(vcd); vcd_lag = np.r_[vcd[0], vcd[:-1]]; sub_lag = np.r_[sub[0], sub[:-1]]
            
            sub_low_idx = np.where(sub < 1.5)[0]; t_since_sub_low = np.zeros(len(sub))
            if len(sub_low_idx) > 0:
                first_low = sub_low_idx[0]; t_since_sub_low[first_low:] = np.arange(first_low, len(sub)) - first_low

            vcd_peak = np.max(vcd); vcd_decay = np.maximum(0, vcd_peak - vcd)

            c["blk"] = b["blk"].copy()
            c["blk"]["vcd_pred"] = vcd[:, None]; c["blk"]["tcd_pred"] = tcd[:, None]; c["blk"]["sub_pred"] = sub[:, None]
            c["blk"]["vcd_growth"] = vcd_growth[:, None]; c["blk"]["cum_vcd"] = cum_vcd[:, None]
            c["blk"]["vcd_lag"] = vcd_lag[:, None]; c["blk"]["sub_lag"] = sub_lag[:, None]
            c["blk"]["time_since_sub_low"] = t_since_sub_low[:, None]; c["blk"]["vcd_decay"] = vcd_decay[:, None]
            out.append(c)
        return out

    train_stacked = inject_stack_features(train, vcd_train, tcd_train, sub_train)
    test_stacked = inject_stack_features(test, vcd_test, tcd_test, sub_test)

    transformed = []
    for i, batch in enumerate(train_stacked):
        altered = batch.copy(); altered["y"] = batch["y"].copy()
        altered["y"][:, 3] = altered["y"][:, 3] / np.maximum(vcd_train[i], 0.1)
        transformed.append(altered)
        
    reg = robust_fused(transformed, test_stacked, 3, BYPRODUCT_FEATS, alpha=3, lam=30, delta=0.10, iterations=4, day_scale=True, clip_inputs=False)
    
    # YENİ: use_weights=True ile MAPE odaklı eğitim
    tree = boosting(transformed, test_stacked, 3, CFG["gfeats"][3], jump_mask(transformed, 3), use_weights=True)
    
    reg_final = reg * vcd_test
    tree_final = tree * vcd_test
    
    return np.maximum((1 - BYPRODUCT_BOOST_SHARE) * reg_final + BYPRODUCT_BOOST_SHARE * tree_final, 1e-6)

## 12. Day-0 byproduct calibration and full pipeline

- `calibrate_p0` shrinks the test day-0 byproduct toward the training mean (`P0_KAPPA`).
- `predict_v14` averages the v9 VCD/TCD with an SVR model. For byproduct it blends the improved model with the original one on days 1-3. If a test batch has day-0 byproduct outside the training range, the improved model is used alone.

In [ ]:
def calibrate_p0(train, test, kappa=P0_KAPPA):
    mu = np.log([b["y0"][3] for b in train]).mean(); out = []
    for b in test:
        c = dict(b); y0 = b["y0"].copy()
        y0[3] = np.exp(mu + kappa * (np.log(y0[3]) - mu))
        c["y0"] = y0; c["blk"] = feature_blocks(c); out.append(c)
    return out


def predict(train, test):
    P = predict_v14(train, test)
    if P0_KAPPA < 1:
        P[..., 3] = predict_v14(train, calibrate_p0(train, test))[..., 3]
    return P


def predict_v14(train, test):
    predictions = predict_v9(train, test)
    
    truth = np.stack([batch["y"][:, :2] for batch in train]); population = np.exp(np.log(truth).mean(axis=0))
    svr = np.empty_like(predictions[..., :2])

    for target in (0, 1):
        names = ["snc" if nm == "snc_tile" else nm for nm in CFG["feats"][target]] + ["fb_now"]
        train_x = np.concatenate([X_of(batch, names) for batch in train]); test_x = np.concatenate([X_of(batch, names) for batch in test])
        scaler = StandardScaler().fit(train_x); keep = ~jump_mask(train, target).ravel()
        log_deviation = np.log(truth[..., target] / population[:, target])
        model = SVR(kernel="rbf", C=0.1, epsilon=0.025, gamma="scale")
        model.fit(scaler.transform(train_x)[keep], log_deviation.ravel()[keep])
        deviation = model.predict(scaler.transform(test_x)).reshape(predictions.shape[:2])
        svr[..., target] = population[:, target] * np.exp(0.30 * np.tanh(deviation / 0.30))

    svr[..., 1] = np.maximum(svr[..., 1], svr[..., 0])
    predictions[..., :2] = 0.5 * (predictions[..., :2] + svr)
    
    byproduct_original = predictions[..., 3].copy()
    
    vcd_train = truth[..., 0]; tcd_train = truth[..., 1]; sub_train = np.stack([batch["y"][:, 2] for batch in train])
    vcd_test = predictions[..., 0]; tcd_test = predictions[..., 1]; sub_test = predictions[..., 2]
    
    byproduct_v12 = improved_byproduct(train, test, vcd_train, tcd_train, sub_train, vcd_test, tcd_test, sub_test)
    
    improved_byproduct_fallback = byproduct_v12.copy()
    byproduct_v12[:, 0] = byproduct_original[:, 0]
    byproduct_v12[:, 1:3] = (0.5 * byproduct_v12[:, 1:3] + 0.5 * byproduct_original[:, 1:3])
    upper_training_initial = max(batch["y0"][3] for batch in train)
    beyond_training_range = np.array([batch["y0"][3] > upper_training_initial for batch in test])
    byproduct_v12[beyond_training_range, :3] = improved_byproduct_fallback[beyond_training_range, :3]
    predictions[..., 3] = byproduct_v12
    return predictions

## 13. Build the submission file

In [ ]:
def make_submission(batches, template_path, out_path):
    train = [b for b in batches if not b["test"]]
    test = [b for b in batches if b["test"]]
    P = predict(train, test)
    tmpl = pd.read_csv(template_path, dtype={"batch_id": str})
    lookup = {b["id"]: (b["t"], p) for b, p in zip(test, P)}
    rows = []
    for _, r in tmpl.iterrows():
        t, p = lookup[r.batch_id]
        k = np.abs(t - float(r.elapsed_time)).argmin()
        assert abs(t[k] - float(r.elapsed_time)) < 1e-6, f"time mismatch in {r.batch_id}"
        rows.append(p[k])
    tmpl[TARGETS] = np.array(rows)
    assert tmpl.shape == (156, 6) and np.isfinite(tmpl[TARGETS].to_numpy(float)).all()
    assert (tmpl[TARGETS].to_numpy(float) > 0).all()
    tmpl.to_csv(out_path, index=False)
    print(f"saved {len(tmpl)} predictions to {out_path}")

## 14. Run

`cross_validate` is not defined in this version, so only the submission file is created.

In [ ]:
batches = prepare(DATA)
make_submission(batches, TEMPLATE, OUTPUT)
pd.read_csv(OUTPUT).head()